In [10]:
import urllib.request
import gzip
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from skimage.feature import hog
import joblib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
def download_mnist_via_keras():
    try:
        import tensorflow as tf
        print("Using TensorFlow/Keras to download MNIST...")
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
        print("Converting Keras format to CSV...")
        print("Converting training data...")
        with open("mnist_train.csv", "w") as f:
            for i in range(len(x_train)):
                label = y_train[i]
                pixels = x_train[i].flatten()
                line = f"{label}," + ",".join(str(p) for p in pixels) + "\n"
                f.write(line)
                if (i + 1) % 10000 == 0:
                    print(f"  Processed {i + 1}/{len(x_train)} training samples")
        print("Converting test data...")
        with open("mnist_test.csv", "w") as f:
            for i in range(len(x_test)):
                label = y_test[i]
                pixels = x_test[i].flatten()
                line = f"{label}," + ",".join(str(p) for p in pixels) + "\n"
                f.write(line)
                if (i + 1) % 2000 == 0:
                    print(f"  Processed {i + 1}/{len(x_test)} test samples")

        print("Conversion complete using Keras method!")
        return True
    except ImportError:
        print("TensorFlow not available for Keras method")
        return False
    except Exception as e:
        print(f"Error with Keras method: {e}")
        return False

def download_and_extract_mnist():
    files = [
        "train-images-idx3-ubyte.gz",
        "train-labels-idx1-ubyte.gz",
        "t10k-images-idx3-ubyte.gz",
        "t10k-labels-idx1-ubyte.gz"
    ]
    base_urls = [
        "https://ossci-datasets.s3.amazonaws.com/mnist/",
        "http://yann.lecun.com/exdb/mnist/",
        "https://github.com/cvdfoundation/mnist/raw/main/",
    ]

    for file in files:
        extracted_file = file[:-3]

        if not os.path.exists(extracted_file):
            print(f"Downloading {file}...")
            downloaded = False
            for base_url in base_urls:
                try:
                    print(f"  Trying {base_url}{file}")
                    urllib.request.urlretrieve(base_url + file, file)
                    downloaded = True
                    break
                except Exception as e:
                    print(f"  Failed: {e}")
                    continue
            if not downloaded:
                print(f"Could not download {file} from any source!")
                print("\nManual download instructions:")
                print("1. Go to http://yann.lecun.com/exdb/mnist/")
                print("2. Download these files manually:")
                print("   - train-images-idx3-ubyte.gz")
                print("   - train-labels-idx1-ubyte.gz")
                print("   - t10k-images-idx3-ubyte.gz")
                print("   - t10k-labels-idx1-ubyte.gz")
                print("3. Extract them using gunzip or any archive tool")
                print("4. Run the converter again")
                return False

            try:
                print(f"Extracting {file}...")
                with gzip.open(file, 'rb') as f_in:
                    with open(extracted_file, 'wb') as f_out:
                        f_out.write(f_in.read())
                os.remove(file)
                print(f"Successfully extracted {extracted_file}")
            #nahi hua so exception
            except Exception as e:
                print(f"Error extracting {file}: {e}")
                return False
        else:
            print(f"{extracted_file} already exists, skipping download")

    return True

def convert(imgf, labelf, outf, n):
    try:
        f = open(imgf, "rb")
        o = open(outf, "w")
        l = open(labelf, "rb")
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Make sure the MNIST files are downloaded first!")
        return False
    f.read(16)
    l.read(8)
    images = []

    print(f"Converting {n} samples from {imgf} and {labelf} to {outf}...")

    for i in range(n):
        image = [ord(l.read(1))]
        for j in range(28*28):
            image.append(ord(f.read(1)))
        images.append(image)
        if (i + 1) % 10000 == 0:
            print(f"Processed {i + 1}/{n} samples")
    for image in images:
        o.write(",".join(str(pix) for pix in image) + "\n")

    f.close()
    o.close()
    l.close()
    print(f"Conversion complete! Saved to {outf}")
    return True
print("Checking for MNIST files...")
success = False
if download_and_extract_mnist():
    print("\nStarting IDX to CSV conversion...")
    if convert("train-images-idx3-ubyte", "train-labels-idx1-ubyte",
               "mnist_train.csv", 60000):

        if convert("t10k-images-idx3-ubyte", "t10k-labels-idx1-ubyte",
                  "mnist_test.csv", 10000):
            success = True

if not success:
    print("\nAutomatic download failed. Trying alternative method...")
    if download_mnist_via_keras():
        success = True

if success:
    print("\nAll conversions completed successfully!")
    print("Files created:")
    print("- mnist_train.csv (60,000 samples)")
    print("- mnist_test.csv (10,000 samples)")
    print("\nYou can now use these files with your TensorFlow training script!")
else:
    print("\nBoth automatic methods failed.")
    print("\nManual download option:")
    print("1. Visit: https://www.kaggle.com/datasets/oddrationale/mnist-in-csv")
    print("2. Download 'mnist_train.csv' and 'mnist_test.csv'")
    print("3. Use them directly with your training script")

Checking for MNIST files...
  Trying https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz
Extracting train-images-idx3-ubyte.gz...
Successfully extracted train-images-idx3-ubyte
  Trying https://ossci-datasets.s3.amazonaws.com/mnist/train-labels-idx1-ubyte.gz
Extracting train-labels-idx1-ubyte.gz...
Successfully extracted train-labels-idx1-ubyte
  Trying https://ossci-datasets.s3.amazonaws.com/mnist/t10k-images-idx3-ubyte.gz
Extracting t10k-images-idx3-ubyte.gz...
Successfully extracted t10k-images-idx3-ubyte
  Trying https://ossci-datasets.s3.amazonaws.com/mnist/t10k-labels-idx1-ubyte.gz
Extracting t10k-labels-idx1-ubyte.gz...
Successfully extracted t10k-labels-idx1-ubyte

Starting IDX to CSV conversion...
Converting 60000 samples from train-images-idx3-ubyte and train-labels-idx1-ubyte to mnist_train.csv...
Processed 10000/60000 samples
Processed 20000/60000 samples
Processed 30000/60000 samples
Processed 40000/60000 samples
Processed 50000/60000 samples
Processed

In [3]:
def load_mnist_csv(path):
    # expects first column label, remaining 784 pixels 0..255
    df = pd.read_csv(path)
    y = df.iloc[:, 0].astype(int).values
    X = df.iloc[:, 1:].values.astype(np.uint8)
    return X, y

print("Loading CSVs...")
X_all, y_all = load_mnist_csv("mnist_train.csv")
X_test_csv, y_test_csv = load_mnist_csv("mnist_test.csv")


Loading CSVs...


In [4]:

# We'll use mnist_train.csv for training / OOF / val; keep mnist_test.csv as final holdout.
# Create holdout split from X_all -> train_full and holdout_val (we will later use stacking and final test)
X_train_full, X_hold_val, y_train_full, y_hold_val = train_test_split(
    X_all, y_all, test_size=10000, stratify=y_all, random_state=42
)
print("Train full:", X_train_full.shape, "Holdout val:", X_hold_val.shape, "Test CSV:", X_test_csv.shape)

Train full: (49999, 784) Holdout val: (10000, 784) Test CSV: (9999, 784)


In [5]:
def extract_hog_features(X, pixels_per_cell=(7,7)):
    # X: (N, 784) uint8
    hog_feats = []
    for img in X:
        arr = img.reshape(28,28)
        feat = hog(arr, orientations=9, pixels_per_cell=pixels_per_cell,
                   cells_per_block=(1,1), feature_vector=True)
        hog_feats.append(feat)
    return np.array(hog_feats)

In [7]:
def train_classical_oof(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=1)
    oof_probs_svc = np.zeros((len(X), 10))
    oof_probs_lr  = np.zeros((len(X), 10))
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"[Classical] Fold {fold+1}/{n_splits}")
        Xtr, ytr = X[tr_idx], y[tr_idx]
        Xval, yval = X[val_idx], y[val_idx]

        # HOG features
        H_tr = extract_hog_features(Xtr)
        H_val = extract_hog_features(Xval)

        # Standard + PCA
        scaler = StandardScaler()
        H_tr_s = scaler.fit_transform(H_tr)
        H_val_s = scaler.transform(H_val)
        pca = PCA(0.98, svd_solver='full')
        H_tr_p = pca.fit_transform(H_tr_s)
        H_val_p = pca.transform(H_val_s)

        # SVC (prob)
        svc = SVC(kernel='rbf', C=10, probability=True, random_state=42)
        svc.fit(H_tr_p, ytr)
        oof_probs_svc[val_idx] = svc.predict_proba(H_val_p)

        # Logistic
        lr = LogisticRegression(max_iter=2000)
        lr.fit(H_tr_p, ytr)
        oof_probs_lr[val_idx] = lr.predict_proba(H_val_p)

    # Save objects for later use on test set: refit on full training set
    print("[Classical] refitting full classical models on entire training set...")
    H_full = extract_hog_features(X)
    scaler_full = StandardScaler().fit(H_full)
    H_full_s = scaler_full.transform(H_full)
    pca_full = PCA(0.98, svd_solver='full').fit(H_full_s)
    H_full_p = pca_full.transform(H_full_s)

    svc_full = SVC(kernel='rbf', C=10, probability=True, random_state=42)
    svc_full.fit(H_full_p, y)

    lr_full = LogisticRegression(max_iter=2000)
    lr_full.fit(H_full_p, y)

    classical_objs = dict(scaler=scaler_full, pca=pca_full, svc=svc_full, lr=lr_full)
    return oof_probs_svc, oof_probs_lr, classical_objs

oof_svc, oof_lr, classical_objs = train_classical_oof(X_train_full, y_train_full, n_splits=5)
print("Classical OOF shapes:", oof_svc.shape, oof_lr.shape)

[Classical] Fold 1/5
[Classical] Fold 2/5
[Classical] Fold 3/5
[Classical] Fold 4/5
[Classical] Fold 5/5
[Classical] refitting full classical models on entire training set...
Classical OOF shapes: (49999, 10) (49999, 10)


In [8]:
def to_images(X):
    return X.reshape(-1,28,28,1).astype('float32')/255.0

# Small CNN builder (parameterize width/depth)
def build_small_cnn(seed=0, width=32, depth=3, dropout=0.4):
    tf.keras.utils.set_random_seed(seed)
    inp = keras.Input((28,28,1))
    x = inp
    for i in range(depth):
        x = layers.Conv2D(width*(i+1), 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPool2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [11]:
def train_cnn_oof(X, y, n_splits=5, epochs=30, batch_size=128):
    X_img = to_images(X)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=2)
    # We will produce OOF probabilities from 3 different CNN variants (different seeds/augs)
    oof_list = [np.zeros((len(X), 10)) for _ in range(3)]
    models_saved = []

    # define augmenters (diverse)
    augs = [
        ImageDataGenerator(width_shift_range=0.08, height_shift_range=0.08, rotation_range=10),
        ImageDataGenerator(width_shift_range=0.12, height_shift_range=0.12, rotation_range=15, shear_range=0.1),
        ImageDataGenerator(),  # no augmentation
    ]

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"[CNN] Fold {fold+1}/{n_splits}")
        Xtr, ytr = X_img[tr_idx], y[tr_idx]
        Xval, yval = X_img[val_idx], y[val_idx]

        for model_i in range(3):
            seed = 100 + model_i  # different seed per model variant
            print(f"  training model_variant {model_i} seed {seed}")
            model = build_small_cnn(seed=seed, width=32 + model_i*8, depth=3, dropout=0.35)
            callbacks = [
                keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, verbose=1),
                keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, verbose=1)
            ]
            aug = augs[model_i]
            model.fit(aug.flow(Xtr, ytr, batch_size=batch_size),
                      epochs=epochs,
                      validation_data=(Xval, yval),
                      callbacks=callbacks,
                      verbose=2)
            # predict val fold
            probs_val = model.predict(Xval, batch_size=256)
            oof_list[model_i][val_idx] = probs_val
            # save fold model file to disk (optional)
            fname = f"cnn_variant{model_i}_fold{fold}.h5"
            model.save(fname)
            models_saved.append(fname)
            keras.backend.clear_session()

    # Refit each variant on full training data (for test predictions)
    print("[CNN] Refitting each variant on full training set (for final test predictions)...")
    full_models = []
    X_full_img = to_images(X)
    for model_i in range(3):
        seed = 100 + model_i
        model = build_small_cnn(seed=seed, width=32 + model_i*8, depth=3, dropout=0.35)
        aug = augs[model_i]
        # fit quickly with EarlyStopping
        model.fit(aug.flow(X_full_img, y, batch_size=batch_size),
                  epochs=epochs, validation_split=0.0,
                  callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)],
                  verbose=2)
        fullname = f"cnn_variant{model_i}_full.h5"
        model.save(fullname)
        full_models.append(fullname)
        keras.backend.clear_session()

    return oof_list, full_models

oof_cnn_list, cnn_full_models = train_cnn_oof(X_train_full, y_train_full, n_splits=5, epochs=20)
print("CNN OOF shapes:", [o.shape for o in oof_cnn_list])
print("Saved full cnn models:", cnn_full_models)

[CNN] Fold 1/5
  training model_variant 0 seed 100
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


313/313 - 21s - 68ms/step - accuracy: 0.8930 - loss: 0.3532 - val_accuracy: 0.4229 - val_loss: 1.6124 - learning_rate: 1.0000e-03
Epoch 2/20
313/313 - 12s - 38ms/step - accuracy: 0.9681 - loss: 0.1052 - val_accuracy: 0.9804 - val_loss: 0.0604 - learning_rate: 1.0000e-03
Epoch 3/20
313/313 - 12s - 37ms/step - accuracy: 0.9770 - loss: 0.0756 - val_accuracy: 0.9863 - val_loss: 0.0419 - learning_rate: 1.0000e-03
Epoch 4/20
313/313 - 10s - 33ms/step - accuracy: 0.9808 - loss: 0.0657 - val_accuracy: 0.9865 - val_loss: 0.0410 - learning_rate: 1.0000e-03
Epoch 5/20
313/313 - 21s - 68ms/step - accuracy: 0.9829 - loss: 0.0582 - val_accuracy: 0.9830 - val_loss: 0.0569 - learning_rate: 1.0000e-03
Epoch 6/20
313/313 - 11s - 35ms/step - accuracy: 0.9854 - loss: 0.0511 - val_accuracy: 0.9904 - val_loss: 0.0362 - learning_rate: 1.0000e-03
Epoch 7/20
313/313 - 11s - 35ms/step - accuracy: 0.9849 - loss: 0.0490 - val_accuracy: 0.9878 - val_loss: 0.0462 - learning_rate: 1.0000e-03
Epoch 8/20
313/313 - 12s

  training model_variant 1 seed 101
Epoch 1/20
313/313 - 20s - 65ms/step - accuracy: 0.8770 - loss: 0.4039 - val_accuracy: 0.5242 - val_loss: 1.7878 - learning_rate: 1.0000e-03
Epoch 2/20
313/313 - 14s - 46ms/step - accuracy: 0.9629 - loss: 0.1281 - val_accuracy: 0.9799 - val_loss: 0.0619 - learning_rate: 1.0000e-03
Epoch 3/20
313/313 - 12s - 39ms/step - accuracy: 0.9720 - loss: 0.0938 - val_accuracy: 0.9775 - val_loss: 0.0816 - learning_rate: 1.0000e-03
Epoch 4/20
313/313 - 12s - 39ms/step - accuracy: 0.9762 - loss: 0.0795 - val_accuracy: 0.9877 - val_loss: 0.0400 - learning_rate: 1.0000e-03
Epoch 5/20
313/313 - 12s - 39ms/step - accuracy: 0.9792 - loss: 0.0707 - val_accuracy: 0.9857 - val_loss: 0.0483 - learning_rate: 1.0000e-03
Epoch 6/20
313/313 - 12s - 37ms/step - accuracy: 0.9824 - loss: 0.0636 - val_accuracy: 0.9885 - val_loss: 0.0376 - learning_rate: 1.0000e-03
Epoch 7/20
313/313 - 20s - 65ms/step - accuracy: 0.9830 - loss: 0.0552 - val_accuracy: 0.9819 - val_loss: 0.0555 - lea

  training model_variant 2 seed 102
Epoch 1/20
313/313 - 13s - 43ms/step - accuracy: 0.9248 - loss: 0.2613 - val_accuracy: 0.4562 - val_loss: 1.5033 - learning_rate: 1.0000e-03
Epoch 2/20
313/313 - 4s - 12ms/step - accuracy: 0.9796 - loss: 0.0705 - val_accuracy: 0.9825 - val_loss: 0.0563 - learning_rate: 1.0000e-03
Epoch 3/20
313/313 - 4s - 12ms/step - accuracy: 0.9863 - loss: 0.0467 - val_accuracy: 0.9879 - val_loss: 0.0440 - learning_rate: 1.0000e-03
Epoch 4/20
313/313 - 4s - 12ms/step - accuracy: 0.9890 - loss: 0.0343 - val_accuracy: 0.9851 - val_loss: 0.0500 - learning_rate: 1.0000e-03
Epoch 5/20
313/313 - 3s - 10ms/step - accuracy: 0.9913 - loss: 0.0280 - val_accuracy: 0.9840 - val_loss: 0.0575 - learning_rate: 1.0000e-03
Epoch 6/20
313/313 - 6s - 19ms/step - accuracy: 0.9916 - loss: 0.0268 - val_accuracy: 0.9896 - val_loss: 0.0406 - learning_rate: 1.0000e-03
Epoch 7/20
313/313 - 3s - 10ms/step - accuracy: 0.9924 - loss: 0.0228 - val_accuracy: 0.9888 - val_loss: 0.0383 - learning_

...

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


391/391 - 15s - 37ms/step - accuracy: 0.9735 - loss: 0.0872
Epoch 3/20
391/391 - 13s - 32ms/step - accuracy: 0.9792 - loss: 0.0686
Epoch 4/20
391/391 - 13s - 32ms/step - accuracy: 0.9819 - loss: 0.0592
Epoch 5/20
391/391 - 13s - 32ms/step - accuracy: 0.9842 - loss: 0.0551
Epoch 6/20
391/391 - 13s - 32ms/step - accuracy: 0.9858 - loss: 0.0495
Epoch 7/20
391/391 - 13s - 32ms/step - accuracy: 0.9870 - loss: 0.0413
Epoch 8/20
391/391 - 13s - 32ms/step - accuracy: 0.9875 - loss: 0.0430
Epoch 9/20
391/391 - 13s - 32ms/step - accuracy: 0.9886 - loss: 0.0382
Epoch 10/20
391/391 - 13s - 32ms/step - accuracy: 0.9894 - loss: 0.0365
Epoch 11/20
391/391 - 13s - 32ms/step - accuracy: 0.9897 - loss: 0.0347
Epoch 12/20
391/391 - 13s - 32ms/step - accuracy: 0.9902 - loss: 0.0344
Epoch 13/20
391/391 - 13s - 32ms/step - accuracy: 0.9899 - loss: 0.0328
Epoch 14/20
391/391 - 13s - 32ms/step - accuracy: 0.9910 - loss: 0.0298
Epoch 15/20
391/391 - 13s - 32ms/step - accuracy: 0.9905 - loss: 0.0315
Epoch 16/20

Epoch 1/20
391/391 - 21s - 55ms/step - accuracy: 0.8933 - loss: 0.3532
Epoch 2/20
391/391 - 13s - 34ms/step - accuracy: 0.9669 - loss: 0.1111
Epoch 3/20
391/391 - 14s - 35ms/step - accuracy: 0.9744 - loss: 0.0851
Epoch 4/20
391/391 - 14s - 35ms/step - accuracy: 0.9787 - loss: 0.0718
Epoch 5/20
391/391 - 13s - 34ms/step - accuracy: 0.9823 - loss: 0.0594
Epoch 6/20
391/391 - 13s - 34ms/step - accuracy: 0.9830 - loss: 0.0575
Epoch 7/20
391/391 - 13s - 34ms/step - accuracy: 0.9839 - loss: 0.0551
Epoch 8/20
391/391 - 14s - 36ms/step - accuracy: 0.9846 - loss: 0.0518
Epoch 9/20
391/391 - 20s - 51ms/step - accuracy: 0.9862 - loss: 0.0463
Epoch 10/20
391/391 - 13s - 35ms/step - accuracy: 0.9864 - loss: 0.0470
Epoch 11/20
391/391 - 13s - 34ms/step - accuracy: 0.9864 - loss: 0.0469
Epoch 12/20
391/391 - 14s - 35ms/step - accuracy: 0.9878 - loss: 0.0406
Epoch 13/20
391/391 - 20s - 51ms/step - accuracy: 0.9884 - loss: 0.0395
Epoch 14/20
391/391 - 14s - 35ms/step - accuracy: 0.9894 - loss: 0.0368
E

Epoch 1/20
391/391 - 11s - 29ms/step - accuracy: 0.9339 - loss: 0.2290
Epoch 2/20
391/391 - 3s - 8ms/step - accuracy: 0.9827 - loss: 0.0590
Epoch 3/20
391/391 - 3s - 8ms/step - accuracy: 0.9862 - loss: 0.0450
Epoch 4/20
391/391 - 3s - 8ms/step - accuracy: 0.9893 - loss: 0.0347
Epoch 5/20
391/391 - 3s - 8ms/step - accuracy: 0.9909 - loss: 0.0284
Epoch 6/20
391/391 - 3s - 8ms/step - accuracy: 0.9918 - loss: 0.0251
Epoch 7/20
391/391 - 3s - 8ms/step - accuracy: 0.9929 - loss: 0.0229
Epoch 8/20
391/391 - 3s - 8ms/step - accuracy: 0.9942 - loss: 0.0180
Epoch 9/20
391/391 - 5s - 13ms/step - accuracy: 0.9934 - loss: 0.0198
Epoch 10/20
391/391 - 3s - 8ms/step - accuracy: 0.9956 - loss: 0.0144
Epoch 11/20
391/391 - 3s - 8ms/step - accuracy: 0.9954 - loss: 0.0146
Epoch 12/20
391/391 - 3s - 8ms/step - accuracy: 0.9958 - loss: 0.0141
Epoch 13/20
391/391 - 3s - 8ms/step - accuracy: 0.9958 - loss: 0.0129
Epoch 14/20
391/391 - 3s - 8ms/step - accuracy: 0.9950 - loss: 0.0156
Epoch 15/20
391/391 - 3s -

CNN OOF shapes: [(49999, 10), (49999, 10), (49999, 10)]
Saved full cnn models: ['cnn_variant0_full.h5', 'cnn_variant1_full.h5', 'cnn_variant2_full.h5']


In [12]:
# ---------- Stack all OOF features and train meta-learner ----------
# Prepare training-level meta features: concatenate classical probs + cnn probs
oof_classical_concat = np.hstack([oof_svc, oof_lr])    # shape (n_samples, 20)
oof_cnn_concat = np.hstack(oof_cnn_list)               # shape (n_samples, 30)
X_meta = np.hstack([oof_classical_concat, oof_cnn_concat])  # (n_samples, 50)
print("Meta features shape:", X_meta.shape)

meta = LogisticRegression(max_iter=2000)
meta.fit(X_meta, y_train_full)
print("Meta trained.")

Meta features shape: (49999, 50)
Meta trained.


In [13]:
# ---------- Final test-time predictions (using saved full models) ----------
# 1) Classical models -> prepare test HOG and predict
print("Preparing classical test predictions...")
H_test = extract_hog_features(X_test_csv)
H_test_s = classical_objs['scaler'].transform(H_test)
H_test_p = classical_objs['pca'].transform(H_test_s)
svc_test_probs = classical_objs['svc'].predict_proba(H_test_p)
lr_test_probs  = classical_objs['lr'].predict_proba(H_test_p)

# 2) CNN models with TTA -> load each full model and apply TTA averaging
def tta_predict_keras(model_path, X_images, augmenter=None, n_augment=8):
    model = keras.models.load_model(model_path)
    if augmenter is None:
        augmenter = ImageDataGenerator(width_shift_range=0.06, height_shift_range=0.06, rotation_range=8)
    preds = []
    N = len(X_images)
    for _ in range(n_augment):
        # generate augmented batch of size N
        batch = next(augmenter.flow(X_images, batch_size=N, shuffle=False))
        preds.append(model.predict(batch, batch_size=256))
    avg = np.mean(preds, axis=0)
    keras.backend.clear_session()
    return avg

Preparing classical test predictions...


In [14]:
print("Running TTA for each CNN full model (this may take time)...")
X_test_img = to_images(X_test_csv)
cnn_test_probs = []
for mp in cnn_full_models:
    probs = tta_predict_keras(mp, X_test_img, n_augment=8)
    cnn_test_probs.append(probs)
cnn_test_concat = np.hstack(cnn_test_probs)  # shape (n_test, 30)

# Combine classical + cnn test probs into meta features
X_test_meta = np.hstack([svc_test_probs, lr_test_probs, cnn_test_concat])
print("Test meta features shape:", X_test_meta.shape)

# Meta predictions
test_pred = meta.predict(X_test_meta)
acc = accuracy_score(y_test_csv, test_pred)
print("Final stacked ensemble test accuracy: {:.5f}".format(acc))
print("Confusion matrix:")
print(confusion_matrix(y_test_csv, test_pred))
print(classification_report(y_test_csv, test_pred))

# Save meta model and arrays
joblib.dump(meta, "meta_logistic.joblib")
np.save("test_meta_features.npy", X_test_meta)
print("Saved meta model and test meta features.")

Running TTA for each CNN full model (this may take time)...
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Test meta features shape: (9999, 50)
Final stacked ensemble test accuracy: 0.99520
Confusion matrix:
[[ 979    0    0    0    0    0    0    1    0    0]
 [   0 1132    1    1    0    0    0    1    0    0]
 [   1    0 1029    0    0    0    0    2    0    0]
 [   0    1    0 1005    0    3    0    0    1    0]
 [   0    0    0    0  978    0    1    0    0    3]
 [   1    0    0    2    0  887    1    0    0    1]
 [   2    1    0    0    0    1  953    0    1    0]
 [   0    2    2    0    0    0    0 1022    1    0]
 [   0    0    2    1    0    1    0    0  970    0]
 [   0    0    0    0    5    2    0    3    3  996]]
              precision    recall  f1-score   support

